# SAM 3 masks for the RealCAD photos (Kaggle or Colab, GPU)

Input : `realcad_all_rgb.zip` (made locally in `out/`): `realcad_all/ids.txt`, `realcad_all/images/<id>/rgb/{1..4}.jpg`
(dev + test, 389 parts; any `realcad_*_rgb.zip` with that layout works)
Output: `realcad_masks.zip` with `images/<id>/mask/{1..4}.png` (0/255, same size as the photo),
`mask_qa.csv` (per photo: prompt used, scores, instances kept, area fraction, flags) and `review/*.jpg` contact sheets.

Before running:
1. Accept the SAM 3 license on https://huggingface.co/facebook/sam3 (gated model) and create a read token.
2. **Kaggle**: Add-ons > Secrets > add `HF_TOKEN`; Settings > Accelerator > GPU T4 x2 (or P100); Internet on.
   Upload the zip as a Kaggle dataset and attach it (Kaggle unzips it; either form is found).
   **Colab**: Runtime > Change runtime type > T4 GPU; key icon > add secret `HF_TOKEN`; the upload cell asks for the zip.
3. Run all. About 1-2 s per photo on a T4, so ~25-50 min for 1556 photos. Re-running skips finished photos.

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate huggingface_hub
import transformers, torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
try:
    from transformers import Sam3Model, Sam3Processor  # noqa: F401
except ImportError:
    print('this transformers release has no SAM 3 -> installing from source (restart the runtime after this cell)')
    !pip install -q -U git+https://github.com/huggingface/transformers

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = False
if not ON_KAGGLE:   # Kaggle images also ship the google.colab package, so only test it off Kaggle
    try:
        import google.colab  # noqa: F401
        ON_COLAB = True
    except ImportError:
        pass
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')

token = os.environ.get('HF_TOKEN')
if not token and ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
if not token and ON_COLAB:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=token) if token else login()   # interactive prompt if no secret was found

In [ ]:
import zipfile, glob

def find_root():
    hits = glob.glob('/kaggle/input/**/ids.txt', recursive=True) + glob.glob(str(WORK / '**/ids.txt'), recursive=True)
    hits = [h for h in hits if (Path(h).parent / 'images').is_dir()]
    return Path(hits[0]).parent if hits else None

ROOT = find_root()
if ROOT is None:
    zips = glob.glob('/kaggle/input/**/realcad_*_rgb.zip', recursive=True) + glob.glob(str(WORK / 'realcad_*_rgb.zip'))
    if not zips and ON_COLAB:
        from google.colab import files
        up = files.upload()                       # pick out/realcad_all_rgb.zip
        zips = [str(WORK / n) for n in up]
    zipfile.ZipFile(zips[0]).extractall(WORK)
    ROOT = find_root()
IDS = [l.strip().split('/')[-1] for l in (ROOT / 'ids.txt').read_text().split() if l.strip()]
OUT = WORK / 'masks_out'
print(ROOT, len(IDS), 'parts')

In [ ]:
from transformers import Sam3Model, Sam3Processor
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Sam3Model.from_pretrained('facebook/sam3').to(dev).eval()
processor = Sam3Processor.from_pretrained('facebook/sam3')

**Mask rule.** Each photo shows one printed part (sometimes in two separate pieces) on a light table.
Prompts are tried in order; the first that finds an instance with score >= `MIN_SCORE` is used.
All instances with score >= `KEEP_FRAC` x best are merged (keeps multi-piece parts), the mask is cleaned
(only pinholes < 0.005 % filled, so through-holes stay background; specks < 0.1 % of the image removed).
If no prompt reaches `MIN_SCORE`, all prompts are retried at `LOW_SCORE` and the best one is kept.
Flags for review: `none` (nothing found), `low` (low-score pass), `multi`
(> 1 instance merged), `tiny` / `huge` (area < 0.5 % or > 40 % of the photo), `border` (mask touches the edge),
`fallback` (not the first prompt).

In [ ]:
import numpy as np, csv, time
from PIL import Image
from scipy import ndimage

PROMPTS = ['3D printed part', 'plastic object', 'object', 'disc', 'ring', 'cylinder', 'plate', 'block']
MIN_SCORE, LOW_SCORE, KEEP_FRAC, MASK_THR = 0.3, 0.12, 0.5, 0.5
HOLE_MAX = 0.00005   # fill only SAM pinholes (< 0.005 % of the photo, ~40 px); real through-holes stay background

@torch.no_grad()
def segment(img, prompt, thr):
    inputs = processor(images=img, text=prompt, return_tensors='pt').to(dev)
    out = model(**inputs)
    r = processor.post_process_instance_segmentation(out, threshold=thr, mask_threshold=MASK_THR,
                                                     target_sizes=inputs.get('original_sizes').tolist())[0]
    return r['masks'].bool().cpu().numpy(), r['scores'].float().cpu().numpy()

def clean(m):
    holes = ndimage.binary_fill_holes(m) & ~m
    lab, n = ndimage.label(holes)
    n_holes = 0
    if n:
        sizes = ndimage.sum(holes, lab, range(1, n + 1))
        m = m | np.isin(lab, 1 + np.nonzero(sizes < HOLE_MAX * m.size)[0])
        n_holes = int((sizes >= HOLE_MAX * m.size).sum())
    lab, n = ndimage.label(m)
    if n > 1:
        sizes = ndimage.sum(m, lab, range(1, n + 1))
        m = np.isin(lab, 1 + np.nonzero(sizes >= 0.001 * m.size)[0])
    return m, n_holes

def mask_photo(img):
    # pass 1: prompts in order at MIN_SCORE; pass 2 (only if nothing found): all prompts at LOW_SCORE, best score wins
    for thr in (MIN_SCORE, LOW_SCORE):
        best = None
        for pi, prompt in enumerate(PROMPTS):
            masks, scores = segment(img, prompt, thr)
            if len(scores) and (best is None or scores.max() > best[2].max()):
                best = (pi, prompt, scores, masks)
                if thr == MIN_SCORE:
                    break
        if best:
            pi, prompt, scores, masks = best
            keep = scores >= KEEP_FRAC * scores.max()
            m, n_holes = clean(masks[keep].any(0))
            return m, {'prompt': prompt, 'fallback': pi > 0, 'low': thr == LOW_SCORE,
                       'best_score': float(scores.max()), 'n_found': int(len(scores)),
                       'n_kept': int(keep.sum()), 'holes': n_holes}
    return np.zeros(img.size[::-1], bool), {'prompt': '', 'fallback': True, 'low': True, 'best_score': 0.0,
                                            'n_found': 0, 'n_kept': 0, 'holes': 0}

QA = WORK / 'mask_qa.csv'
done = set()
if QA.exists():
    done = {(r['id'], r['view']) for r in csv.DictReader(open(QA))}
fields = ['id', 'view', 'prompt', 'best_score', 'n_found', 'n_kept', 'holes', 'area_frac', 'flags']
new = not QA.exists()
f = open(QA, 'a', newline=''); w = csv.DictWriter(f, fields)
if new: w.writeheader()
t0, n = time.time(), 0
for i, pid in enumerate(IDS):
    for k in (1, 2, 3, 4):
        if (pid, str(k)) in done:
            continue
        img = Image.open(ROOT / 'images' / pid / 'rgb' / f'{k}.jpg').convert('RGB')
        m, info = mask_photo(img)
        dst = OUT / 'images' / pid / 'mask' / f'{k}.png'
        dst.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray((m * 255).astype(np.uint8)).save(dst)
        area = float(m.mean())
        flags = [x for x, c in [('none', info['n_found'] == 0), ('multi', info['n_kept'] > 1),
                                ('tiny', 0 < area < 0.005), ('huge', area > 0.4),
                                ('border', m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any()),
                                ('fallback', info['fallback'] and info['n_found'] > 0),
                                ('low', info['low'] and info['n_found'] > 0)] if c]
        w.writerow({'id': pid, 'view': k, 'prompt': info['prompt'], 'best_score': round(info['best_score'], 3),
                    'n_found': info['n_found'], 'n_kept': info['n_kept'], 'holes': info['holes'],
                    'area_frac': round(area, 4),
                    'flags': ' '.join(flags)})
        n += 1
    f.flush()
    if (i + 1) % 20 == 0:
        print(f'{i + 1}/{len(IDS)} parts  {(time.time() - t0) / max(n, 1):.2f} s/photo', flush=True)
f.close()
print('done', n, 'new photos')

In [ ]:
# contact sheets: every part, 4 views, mask outline in red; flagged photos get a red frame
from collections import Counter
from PIL import ImageDraw
qa = list(csv.DictReader(open(QA)))
print(len(qa), 'photos; flags:', dict(Counter(x for r in qa for x in r['flags'].split())))
flagged = {(r['id'], int(r['view'])) for r in qa if r['flags']}
REV = WORK / 'review'; REV.mkdir(exist_ok=True)
T = 160
def thumb(pid, k):
    img = Image.open(ROOT / 'images' / pid / 'rgb' / f'{k}.jpg').convert('RGB')
    m = np.array(Image.open(OUT / 'images' / pid / 'mask' / f'{k}.png')) > 0
    edge = m ^ ndimage.binary_erosion(m, iterations=3)
    a = np.array(img); a[edge] = [255, 0, 0]
    t = Image.fromarray(a); t.thumbnail((T, T))
    if (pid, k) in flagged:
        ImageDraw.Draw(t).rectangle([0, 0, t.width - 1, t.height - 1], outline=(255, 0, 0), width=4)
    return t
per = 12
for s in range(0, len(IDS), per):
    chunk = IDS[s:s + per]
    sheet = Image.new('RGB', (4 * T + 120, len(chunk) * (T * 3 // 4 + 4)), 'white')
    d = ImageDraw.Draw(sheet)
    for r, pid in enumerate(chunk):
        y = r * (T * 3 // 4 + 4)
        d.text((2, y + 2), pid, fill='black')
        for k in (1, 2, 3, 4):
            sheet.paste(thumb(pid, k), (120 + (k - 1) * T, y))
    sheet.save(REV / f'sheet_{s // per:03d}.jpg', quality=85)
print('sheets in', REV)

In [ ]:
# package the results for download
import shutil
bundle = WORK / 'realcad_masks'
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(OUT, bundle)
shutil.copy(QA, bundle / 'mask_qa.csv')
shutil.copytree(REV, bundle / 'review')
shutil.make_archive(str(bundle), 'zip', bundle)
print(bundle.with_suffix('.zip'), round(bundle.with_suffix('.zip').stat().st_size / 1e6, 1), 'MB')
if ON_COLAB:
    from google.colab import files
    files.download(str(bundle.with_suffix('.zip')))
if ON_KAGGLE:   # clickable download link
    from IPython.display import FileLink, display
    os.chdir(WORK); display(FileLink(bundle.with_suffix('.zip').name))